# Diboson bad-file scan

Scan the skimmed WW, WZ, and ZZ samples before production processing.

A file is marked bad when:

- the ROOT file or `Events` tree cannot be opened,
- any required event-filter branch is missing, or
- reading the required branches through the full tree raises an exception.

`skim_factor` is currently a temporary value in `backgrounds.yaml`; do not interpret absolute normalized yields until it is measured.

In [1]:
import os
import sys
import importlib
from concurrent.futures import ThreadPoolExecutor, as_completed

import awkward as ak
import uproot

sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities

importlib.reload(utilities)

<module 'sidm.tools.utilities' from '/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/tools/utilities.py'>

In [2]:
samples = ["WW", "WZ", "ZZ"]

fileset = utilities.make_fileset(
    samples,
    "skimmed_llpNanoAOD_v2",
    location_cfg="backgrounds.yaml",
)

for sample in samples:
    print(f"{sample}: {len(fileset[sample]['files'])} files")
    print(f"  first: {fileset[sample]['files'][0]}")

WW: 999 files
  first: root://xcache//store/group/lpcmetx/SIDM/Backgrounds/2018_v2/Skims/WW_TuneCP5_13TeV_pythia8/skimmed_output_1.root
WZ: 730 files
  first: root://xcache//store/group/lpcmetx/SIDM/Backgrounds/2018_v2/Skims/WZ_TuneCP5_13TeV_pythia8/skimmed_output_1.root
ZZ: 338 files
  first: root://xcache//store/group/lpcmetx/SIDM/Backgrounds/2018_v2/Skims/ZZ_TuneCP5_13TeV_pythia8/skimmed_output_1.root


In [3]:
FLAG_BRANCHES = [
    "Flag_goodVertices",
    "Flag_globalSuperTightHalo2016Filter",
    "Flag_HBHENoiseFilter",
    "Flag_HBHENoiseIsoFilter",
    "Flag_EcalDeadCellTriggerPrimitiveFilter",
    "Flag_BadPFMuonFilter",
    "Flag_BadPFMuonDzFilter",
    "Flag_eeBadScFilter",
    "Flag_ecalBadCalibFilter",
    "Flag_hfNoisyHitsFilter",
]


def check_file(fname, treename="Events", step_size="50 MB"):
    try:
        with uproot.open(fname, timeout=60) as root_file:
            tree = root_file[treename]
            tree_keys = set(tree.keys())
            missing = sorted(set(FLAG_BRANCHES) - tree_keys)
            if missing:
                return {
                    "file": fname,
                    "status": "bad",
                    "entries": tree.num_entries,
                    "missing": missing,
                    "error": "Missing required branches",
                }

            for _ in tree.iterate(FLAG_BRANCHES, step_size=step_size, library="ak"):
                pass

            return {
                "file": fname,
                "status": "ok",
                "entries": tree.num_entries,
                "missing": [],
                "error": None,
            }
    except Exception as err:
        return {
            "file": fname,
            "status": "bad",
            "entries": None,
            "missing": None,
            "error": repr(err),
        }

In [4]:
max_workers = 32

scan_results = {}
bad_files = {}

for sample in samples:
    files = fileset[sample]["files"]
    sample_results = []
    sample_bad_files = []

    print(f"\nScanning {sample}: {len(files)} files")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(check_file, fname) for fname in files]
        for idx, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            sample_results.append(result)

            if result["status"] == "bad":
                print("BAD:", result["file"])
                if result["missing"]:
                    print("  missing:", result["missing"])
                print("  error:", result["error"])
                sample_bad_files.append(result["file"])

            if idx % 100 == 0 or idx == len(files):
                print(f"checked {idx}/{len(files)} files; bad={len(sample_bad_files)}")

    scan_results[sample] = sample_results
    bad_files[sample] = sample_bad_files


Scanning WW: 999 files
checked 100/999 files; bad=0
checked 200/999 files; bad=0
checked 300/999 files; bad=0
checked 400/999 files; bad=0
checked 500/999 files; bad=0
checked 600/999 files; bad=0
checked 700/999 files; bad=0
checked 800/999 files; bad=0
checked 900/999 files; bad=0
checked 999/999 files; bad=0

Scanning WZ: 730 files
checked 100/730 files; bad=0
checked 200/730 files; bad=0
checked 300/730 files; bad=0
checked 400/730 files; bad=0
checked 500/730 files; bad=0
checked 600/730 files; bad=0
checked 700/730 files; bad=0
checked 730/730 files; bad=0

Scanning ZZ: 338 files
checked 100/338 files; bad=0
checked 200/338 files; bad=0
checked 300/338 files; bad=0
checked 338/338 files; bad=0


In [5]:
for sample in samples:
    results = scan_results[sample]
    good = sum(result["status"] == "ok" for result in results)
    bad = len(bad_files[sample])
    entries = sum(result["entries"] or 0 for result in results if result["status"] == "ok")
    print(f"{sample}: total={len(results)}, good={good}, bad={bad}, readable entries={entries}")

WW: total=999, good=999, bad=0, readable entries=2318370
WZ: total=730, good=730, bad=0, readable entries=1257593
ZZ: total=338, good=338, bad=0, readable entries=414939


In [ ]:
clean_fileset = {
    sample: {
        **fileset[sample],
        "files": [
            fname
            for fname in fileset[sample]["files"]
            if fname not in bad_files[sample]
        ],
    }
    for sample in samples
}

for sample in samples:
    print(
        f"{sample}: original={len(fileset[sample]['files'])}, "
        f"clean={len(clean_fileset[sample]['files'])}"
    )

In [ ]:
for sample in samples:
    print(f"\n{sample} bad files ({len(bad_files[sample])}):")
    for fname in bad_files[sample]:
        print(fname)